# ⚡️ Energy-Based Models

In this notebook, we'll walk through the steps required to train your own Energy Based Model to predict the distribution of a demo dataset

The code is adapted from the excellent ['Deep Energy-Based Generative Models' tutorial](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial8/Deep_Energy_Models.html) created by Phillip Lippe.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import os
os.environ["KERAS_BACKEND"] = "torch"  # Keras only loads the dataset; this keeps TensorFlow out of the PyTorch process
from keras import datasets

import sys
from pathlib import Path

# Make the repo root importable so `notebooks.utils_pytorch` can be found
REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils_pytorch import display, sample_batch
import random

# Select the device - PyTorch needs to be told where to run
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # Apple Silicon
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 32
CHANNELS = 1
STEP_SIZE = 10
STEPS = 60
NOISE = 0.005
ALPHA = 0.1
GRADIENT_CLIP = 0.03
BATCH_SIZE = 128
BUFFER_SIZE = 8192
LEARNING_RATE = 0.0001
EPOCHS = 60
LOAD_MODEL = False

In [ ]:
# Load the data - Keras downloads it once (cached in ~/.keras/datasets) and returns
# NumPy arrays of shape (N, 28, 28) uint8; we don't need the labels
(x_train, _), (x_test, _) = datasets.mnist.load_data()

In [ ]:
# Preprocess the data


def preprocess(imgs):
    """
    Normalize and reshape the images
    """
    imgs = (imgs.astype("float32") - 127.5) / 127.5
    imgs = np.pad(imgs, ((0, 0), (2, 2), (2, 2)), constant_values=-1.0)
    imgs = np.expand_dims(imgs, 1)  # channels first: (N, 1, 32, 32)
    return torch.tensor(imgs)


x_train = preprocess(x_train)
x_test = preprocess(x_test)

In [ ]:
# A DataLoader can batch a tensor of images directly. drop_last=True keeps every
# training batch at BATCH_SIZE, the same size as the batch of fake images
x_train = DataLoader(x_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
x_test = DataLoader(x_test, batch_size=BATCH_SIZE)

In [ ]:
# Show some items of clothing from the training set
train_sample = sample_batch(x_train)
display(train_sample)

## 2. Build the EBM network <a name="train"></a>

In [ ]:
# Keras "same" padding with stride 2 halves the image size; in PyTorch we pick
# padding = kernel_size // 2 to get the same output shapes (32 -> 16 -> 8 -> 4 -> 2)
model = nn.Sequential(
    nn.Conv2d(CHANNELS, 16, kernel_size=5, stride=2, padding=2),
    nn.SiLU(),  # SiLU is PyTorch's name for swish
    nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
    nn.SiLU(),
    nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
    nn.SiLU(),
    nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
    nn.SiLU(),
    nn.Flatten(),
    nn.Linear(64 * 2 * 2, 64),
    nn.SiLU(),
    nn.Linear(64, 1),  # one score per image: high = realistic (low energy)
).to(device)

print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
if LOAD_MODEL:
    model.load_state_dict(torch.load("./models/model.pt", map_location=device))

## 2. Set up a Langevin sampler function <a name="sampler"></a>

In [ ]:
# Function to generate samples using Langevin Dynamics
def generate_samples(
    model, inp_imgs, steps, step_size, noise, return_img_per_step=False
):
    imgs_per_step = []
    inp_imgs = inp_imgs.to(next(model.parameters()).device)
    with torch.enable_grad():  # we need gradients w.r.t. the images, even inside no_grad code
        for _ in range(steps):
            inp_imgs = inp_imgs + torch.randn_like(inp_imgs) * noise
            inp_imgs = inp_imgs.clamp(-1.0, 1.0).detach().requires_grad_(True)
            out_score = model(inp_imgs)
            # Gradient of the score w.r.t. the input pixels (not the weights)
            (grads,) = torch.autograd.grad(out_score.sum(), inp_imgs)
            grads = grads.clamp(-GRADIENT_CLIP, GRADIENT_CLIP)
            # Step uphill on the score, i.e. towards lower energy
            inp_imgs = (inp_imgs + step_size * grads).clamp(-1.0, 1.0).detach()
            if return_img_per_step:
                imgs_per_step.append(inp_imgs)
    if return_img_per_step:
        return torch.stack(imgs_per_step, dim=0)
    else:
        return inp_imgs

## 3. Set up a buffer to store examples <a name="buffer"></a>

In [ ]:
class Buffer:
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.examples = [
            torch.rand(1, CHANNELS, IMAGE_SIZE, IMAGE_SIZE) * 2 - 1
            for _ in range(BATCH_SIZE)
        ]

    def sample_new_exmps(self, steps, step_size, noise):
        # ~5% fresh random noise, the rest restarted from previous samples
        n_new = np.random.binomial(BATCH_SIZE, 0.05)
        rand_imgs = torch.rand(n_new, CHANNELS, IMAGE_SIZE, IMAGE_SIZE) * 2 - 1
        old_imgs = torch.cat(
            random.choices(self.examples, k=BATCH_SIZE - n_new), dim=0
        )
        inp_imgs = torch.cat([rand_imgs, old_imgs], dim=0)
        inp_imgs = generate_samples(
            self.model, inp_imgs, steps=steps, step_size=step_size, noise=noise
        )
        # The buffer itself lives on the CPU
        self.examples = list(torch.split(inp_imgs.cpu(), 1, dim=0)) + self.examples
        self.examples = self.examples[:BUFFER_SIZE]
        return inp_imgs

In [ ]:
class EBM(nn.Module):
    """Holds the energy model, its sample buffer and the train / test steps."""

    def __init__(self, model):
        super().__init__()
        self.model = model
        self.buffer = Buffer(self.model)
        self.alpha = ALPHA
        self.optimizer = None  # set in the "compile" cell below

    def train_step(self, real_imgs):
        real_imgs = real_imgs + torch.randn_like(real_imgs) * NOISE
        real_imgs = real_imgs.clamp(-1.0, 1.0)
        fake_imgs = self.buffer.sample_new_exmps(
            steps=STEPS, step_size=STEP_SIZE, noise=NOISE
        )
        inp_imgs = torch.cat([real_imgs, fake_imgs], dim=0)
        real_out, fake_out = torch.chunk(self.model(inp_imgs), 2, dim=0)
        # Contrastive divergence: push real scores up and fake scores down
        cdiv_loss = fake_out.mean() - real_out.mean()
        # Keep the scores close to zero so they don't drift off
        reg_loss = self.alpha * (real_out**2 + fake_out**2).mean()
        loss = cdiv_loss + reg_loss

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return {
            "loss": loss.item(),
            "reg": reg_loss.item(),
            "cdiv": cdiv_loss.item(),
            "real": real_out.mean().item(),
            "fake": fake_out.mean().item(),
        }

    @torch.no_grad()
    def test_step(self, real_imgs):
        fake_imgs = torch.rand_like(real_imgs) * 2 - 1
        inp_imgs = torch.cat([real_imgs, fake_imgs], dim=0)
        real_out, fake_out = torch.chunk(self.model(inp_imgs), 2, dim=0)
        cdiv = fake_out.mean() - real_out.mean()
        return {
            "cdiv": cdiv.item(),
            "real": real_out.mean().item(),
            "fake": fake_out.mean().item(),
        }

In [ ]:
ebm = EBM(model)

## 3. Train the EBM network <a name="train"></a>

In [ ]:
# Keras compile() -> just create the optimizer
ebm.optimizer = torch.optim.Adam(ebm.model.parameters(), lr=LEARNING_RATE)

In [ ]:
writer = SummaryWriter(log_dir="./logs")  # TensorBoard logging


class ImageGenerator:
    """Plain Python version of the Keras callback; the training loop calls on_epoch_end."""

    def __init__(self, num_img):
        self.num_img = num_img

    def on_epoch_end(self, epoch):
        start_imgs = (
            torch.rand(self.num_img, CHANNELS, IMAGE_SIZE, IMAGE_SIZE) * 2 - 1
        )
        generated_images = generate_samples(
            ebm.model,
            start_imgs,
            steps=1000,
            step_size=STEP_SIZE,
            noise=NOISE,
            return_img_per_step=False,
        )
        display(
            generated_images,
            save_to="./output/generated_img_%03d.png" % (epoch),
        )

        example_images = torch.cat(random.choices(ebm.buffer.examples, k=10), dim=0)
        display(
            example_images, save_to="./output/example_img_%03d.png" % (epoch)
        )


image_generator_callback = ImageGenerator(num_img=10)

In [ ]:
class SaveModel:
    def on_epoch_end(self, epoch):
        torch.save(model.state_dict(), "./models/model.pt")


save_model_callback = SaveModel()

In [ ]:
for epoch in range(EPOCHS):
    # Training: average each metric over the epoch
    ebm.train()
    train_logs = []
    for real_imgs in x_train:
        train_logs.append(ebm.train_step(real_imgs.to(device)))
    train_logs = {k: np.mean([log[k] for log in train_logs]) for k in train_logs[0]}

    # Validation: real test images vs. pure random noise
    ebm.eval()
    val_logs = [ebm.test_step(real_imgs.to(device)) for real_imgs in x_test]
    val_logs = {"val_" + k: np.mean([log[k] for log in val_logs]) for k in val_logs[0]}

    logs = {**train_logs, **val_logs}
    print(f"Epoch {epoch + 1}/{EPOCHS} - " + " - ".join(f"{k}: {v:.4f}" for k, v in logs.items()))
    for k, v in logs.items():
        writer.add_scalar(k, v, epoch)

    save_model_callback.on_epoch_end(epoch)
    image_generator_callback.on_epoch_end(epoch)

writer.close()

## 4. Generate images <a name="generate"></a>

In [ ]:
start_imgs = torch.rand(10, CHANNELS, IMAGE_SIZE, IMAGE_SIZE) * 2 - 1

In [ ]:
display(start_imgs)

In [ ]:
gen_img = generate_samples(
    ebm.model,
    start_imgs,
    steps=1000,
    step_size=STEP_SIZE,
    noise=NOISE,
    return_img_per_step=True,
)

In [ ]:
display(gen_img[-1])

In [ ]:
# gen_img has shape (steps, num_images, C, H, W): follow image 6 through time
imgs = []
for i in [0, 1, 3, 5, 10, 30, 50, 100, 300, 999]:
    imgs.append(gen_img[i][6])

display(torch.stack(imgs))